# 01 — Extração de Dados Brutos

**Objetivo:** baixar e armazenar todos os dados brutos necessários para o projeto, sem aplicar transformações ou filtros. As limpezas, classificações e merges acontecem no notebook `02_transformacao.ipynb`.

## Fontes coletadas

| # | Fonte | Acesso | Tabela / Endpoint | Conteúdo |
|---|-------|--------|-------------------|----------|
| 1 | **SIM-DATASUS** | FTP DATASUS | `/dissemin/publicos/SIM/CID10/DORES/` | Microdados de óbitos por UF/ano (DO\*.dbc) |
| 2 | **População Municipal** | IBGE Sidra | Tab 4714 var 93 (Censo 2022) / Tab 6579 var 9324 (estimativas) | População residente por município/ano |
| 3 | **PIB Municipal** | IBGE Sidra | Tab 5938 var 37 | PIB total a preços correntes (R$ mil) |
| 4 | **Despesa em Saúde** | SICONFI / Tesouro Nacional | API DCA-Anexo I-E, função 10 (Saúde) | Despesas empenhadas/liquidadas em saúde por município |
| 5 | **IDH-M, Gini, RDPC** | IPEA | API Ipea | Indicadores socioeconômicos do Censo 2010 |

## Saída

```
data/raw/
├── sim/                              # 1 parquet por UF/ano (DBC original convertido)
│   ├── DOAC2022.parquet
│   └── ...
├── ibge/
│   ├── populacao_municipal_{ano}.csv
│   └── pib_municipal_{ano}.csv
├── siconfi/
│   └── despesa_saude_{ano}.csv       # consolidado de todos os municípios
└── atlas/
    └── atlas_brasil.csv              # download manual
```

In [1]:
import ftplib
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from datasus_dbc import decompress as dbc_decompress
from dbfread import DBF

ROOT = Path.cwd().parent
RAW_DIR      = ROOT / "data" / "raw"
SIM_DIR      = RAW_DIR / "sim"
IBGE_DIR     = RAW_DIR / "ibge"
SICONFI_DIR  = RAW_DIR / "siconfi"
ATLAS_DIR    = RAW_DIR / "atlas"
for d in (SIM_DIR, IBGE_DIR, SICONFI_DIR, ATLAS_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
print("Imports OK")

Imports OK


## 1. Configuração de escopo

Defina o conjunto de UFs e anos a extrair. **Todas as etapas são idempotentes** — re-executar só baixa o que falta.

| Parâmetro | Valor recomendado | Observação |
|-----------|-------------------|------------|
| `ANOS`    | `[2022, 2023, 2024]` | Painel de 3 anos; SIM tem cobertura completa nos três |
| `UFS`     | `UFS_TODAS`       | Todas as 27 UFs para análise nacional |

**Lacunas conhecidas nas fontes (tratadas com fallback automático abaixo):**

| Fonte | Ano | Problema | Fallback aplicado |
|-------|-----|----------|-------------------|
| População IBGE (Tab 6579 var 9324) | 2023 | IBGE não publicou estimativa para 2023 (ano do Censo + 1) | Usa Censo 2022 (Tab 4714) como proxy |
| PIB Municipal IBGE (Tab 5938 var 37) | 2024 | Lag estrutural de ~2 anos | Usa PIB 2023 como proxy |
| SICONFI Despesa em Saúde | 2024 | Municípios ainda submetendo balanço anual | Aceita cobertura parcial (não bloqueia) |
| Atlas DH (IPEA) | 2022-2024 | Atlas só foi calculado com base no Censo 2010 | Indicadores tratados como estruturais (constantes no período) |

O arquivo salvo em disco usa o **ano requisitado** no nome, e uma coluna `ANO_FONTE` registra a origem real do dado para auditoria.

In [2]:
UFS_TODAS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO",
    "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR",
    "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO",
]

ANOS = [2022, 2023 , 2024]
UFS  = UFS_TODAS

print(f"ANOS : {ANOS}")
print(f"UFS  : {len(UFS)} estados")

ANOS : [2022, 2023, 2024]
UFS  : 27 estados


## 2. SIM-DATASUS — Microdados de óbitos

Download dos arquivos `DO{UF}{ANO}.dbc` direto do FTP DATASUS, conversão para Parquet sem nenhuma transformação. Cada arquivo é salvo em `data/raw/sim/`.

In [3]:
FTP_HOST = "ftp.datasus.gov.br"
FTP_DIR  = "/dissemin/publicos/SIM/CID10/DORES"


def baixar_dbcs_sim(ufs, anos, destino: Path) -> list[Path]:
    """Baixa os DOXXAAAA.dbc do FTP reaproveitando uma única conexão."""
    pendentes, cacheados = [], []
    for uf in ufs:
        for ano in anos:
            nome = f"DO{uf.upper()}{ano}.dbc"
            local = destino / nome
            (cacheados if local.exists() else pendentes).append((nome, local))
    print(f"Em cache: {len(cacheados)}  |  A baixar: {len(pendentes)}")
    if pendentes:
        with ftplib.FTP(FTP_HOST, timeout=180) as ftp:
            ftp.login()
            ftp.cwd(FTP_DIR)
            for i, (nome, local) in enumerate(pendentes, 1):
                print(f"  [{i:>2}/{len(pendentes)}] {nome} ...", end=" ", flush=True)
                try:
                    with open(local, "wb") as f:
                        ftp.retrbinary(f"RETR {nome}", f.write)
                    print(f"{local.stat().st_size / 1024:.0f} KB")
                except ftplib.error_perm as e:
                    local.unlink(missing_ok=True)
                    print(f"FALHOU: {e}")
    return [destino / f"DO{uf}{ano}.dbc"
            for uf in ufs for ano in anos
            if (destino / f"DO{uf}{ano}.dbc").exists()]


dbc_files = baixar_dbcs_sim(UFS, ANOS, SIM_DIR)
print(f"\nTotal de DBCs disponíveis: {len(dbc_files)}")

Em cache: 81  |  A baixar: 0

Total de DBCs disponíveis: 81


In [4]:
def dbc_para_parquet(dbc_path: Path) -> Path:
    """
    Converte DBC → DBF (temporário) → DataFrame → Parquet (raw, sem transformações).
    Idempotente: pula se o parquet já existe.
    """
    parquet_path = dbc_path.with_suffix(".parquet")
    if parquet_path.exists():
        return parquet_path

    dbf_path = dbc_path.with_suffix(".dbf")
    dbc_decompress(str(dbc_path), str(dbf_path))
    try:
        table = DBF(str(dbf_path), encoding="latin1", char_decode_errors="replace")
        df = pd.DataFrame(iter(table))
    finally:
        dbf_path.unlink(missing_ok=True)

    df.to_parquet(parquet_path, index=False)
    return parquet_path


for i, dbc in enumerate(dbc_files, 1):
    pq = dbc_para_parquet(dbc)
    n = len(pd.read_parquet(pq, columns=[pd.read_parquet(pq).columns[0]]))
    print(f"  [{i:>2}/{len(dbc_files)}] {pq.name}  → {n:,} registros")

print(f"\n SIM em: {SIM_DIR}")

  [ 1/81] DOAC2022.parquet  → 4,159 registros
  [ 2/81] DOAC2023.parquet  → 4,189 registros


  [ 3/81] DOAC2024.parquet  → 4,267 registros


  [ 4/81] DOAL2022.parquet  → 23,122 registros
  [ 5/81] DOAL2023.parquet  → 21,965 registros
  [ 6/81] DOAL2024.parquet  → 22,653 registros
  [ 7/81] DOAM2022.parquet  → 20,155 registros


  [ 8/81] DOAM2023.parquet  → 20,385 registros


  [ 9/81] DOAM2024.parquet  → 20,225 registros
  [10/81] DOAP2022.parquet  → 3,846 registros


...

  [61/81] DORO2022.parquet  → 10,277 registros
  [62/81] DORO2023.parquet  → 9,974 registros
  [63/81] DORO2024.parquet  → 10,097 registros
  [64/81] DORR2022.parquet  → 3,246 registros
  [65/81] DORR2023.parquet  → 3,311 registros
  [66/81] DORR2024.parquet  → 3,055 registros
  [67/81] DORS2022.parquet  → 104,096 registros


  [68/81] DORS2023.parquet  → 93,516 registros
  [69/81] DORS2024.parquet  → 101,480 registros
  [70/81] DOSC2022.parquet  → 51,317 registros
  [71/81] DOSC2023.parquet  → 48,589 registros


  [72/81] DOSC2024.parquet  → 52,224 registros
  [73/81] DOSE2022.parquet  → 14,791 registros
  [74/81] DOSE2023.parquet  → 14,087 registros
  [75/81] DOSE2024.parquet  → 14,453 registros


  [76/81] DOSP2022.parquet  → 354,056 registros
  [77/81] DOSP2023.parquet  → 334,303 registros


  [78/81] DOSP2024.parquet  → 351,616 registros
  [79/81] DOTO2022.parquet  → 9,370 registros
  [80/81] DOTO2023.parquet  → 8,912 registros
  [81/81] DOTO2024.parquet  → 9,427 registros

 SIM em: C:\Users\cerb3\OneDrive\Área de Trabalho\IESB\Tópicos BD\data\raw\sim


## 3. População Municipal — IBGE Sidra

Fonte por ano:

| Ano | Tabela | Variável | Tipo |
|-----|--------|----------|------|
| 2022 | 4714 | 93 | Censo Demográfico |
| 2023 | — | — | **Sem dado**. Fallback: Censo 2022 (Tab 4714) |
| 2024 | 6579 | 9324 | Estimativa populacional |

Cada chamada é cacheada em `data/raw/ibge/populacao_municipal_{ano}.csv`. A coluna `ANO` é o ano *requisitado* (preserva o alinhamento com SIM); `ANO_FONTE` registra a origem real do dado.

In [5]:
# Fontes por ano: (tabela, variável, ano_da_fonte). ano_da_fonte != ano requisitado quando há fallback.
POP_FONTES = {
    2022: ("4714", "93",   2022),   # Censo
    2023: ("4714", "93",   2022),   # FALLBACK: IBGE pulou estimativa de 2023; usa Censo 2022
    2024: ("6579", "9324", 2024),   # Estimativa
    2025: ("6579", "9324", 2025),   # (reserva)
}


def _sidra_get(url: str) -> list[dict]:
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    raw = r.json()
    # Sidra devolve len==1 (só metadado) quando o ano não tem dado
    return raw if len(raw) >= 2 else []


def fetch_populacao_municipal(ano: int) -> pd.DataFrame:
    cache = IBGE_DIR / f"populacao_municipal_{ano}.csv"
    if cache.exists():
        return pd.read_csv(cache, dtype={"CODMUN7": str, "CODMUN6": str})

    if ano not in POP_FONTES:
        raise ValueError(f"Ano {ano} não mapeado em POP_FONTES")

    tab, var, ano_fonte = POP_FONTES[ano]
    url = f"https://apisidra.ibge.gov.br/values/t/{tab}/n6/all/v/{var}/p/{ano_fonte}"
    aviso = "" if ano_fonte == ano else f"  ⚠️  ano {ano} sem dado direto — usando fonte de {ano_fonte}"
    print(f"  → Sidra: {url}{aviso}")

    raw = _sidra_get(url)
    if not raw:
        raise RuntimeError(f"Sidra retornou vazio para população {ano} (tab {tab}, p={ano_fonte})")

    df = pd.DataFrame(raw[1:])
    df = df.rename(columns={"D1C": "CODMUN7", "D1N": "MUNICIPIO_NOME", "V": "POPULACAO"})
    df["POPULACAO"]   = pd.to_numeric(df["POPULACAO"], errors="coerce")
    df["ANO"]         = ano           # ano de referência (alinha com SIM)
    df["ANO_FONTE"]   = ano_fonte     # origem real do dado
    df["CODMUN7"]     = df["CODMUN7"].astype(str).str.strip()
    df["CODMUN6"]     = df["CODMUN7"].str[:6]
    df = df[["CODMUN7", "CODMUN6", "MUNICIPIO_NOME", "ANO", "ANO_FONTE", "POPULACAO"]]
    df.to_csv(cache, index=False)
    print(f"     {len(df):,} municípios | população total: {df['POPULACAO'].sum():,.0f}")
    return df


for ano in ANOS:
    fetch_populacao_municipal(ano)

print(f"\nPopulação em: {IBGE_DIR}")


População em: C:\Users\cerb3\OneDrive\Área de Trabalho\IESB\Tópicos BD\data\raw\ibge


## 3.1. População por faixa etária — IBGE Sidra Tabela 9514 (Censo 2022)

Para **padronizar a taxa de mortalidade por idade** (notebook 06), precisamos do denominador por faixa etária: quantas pessoas vivas de cada idade há em cada município. A tabela 9514 do Censo 2022 fornece população por município × sexo × idade.

Extraímos as 14 faixas quinquenais de 5 a 74 anos (total de sexos). A API limita 50.000 valores por requisição (5.570 municípios × 14 faixas ≈ 78.000), então a busca é feita em **dois lotes de 7 faixas**. Saída: `data/raw/ibge/populacao_idade_municipal_2022.csv`.

In [6]:
# Faixas quinquenais 5-74 e códigos na classificação c287 do Sidra (tabela 9514)
GRUPOS_IDADE = {
    93084: "05-09", 93085: "10-14", 93086: "15-19", 93087: "20-24", 93088: "25-29",
    93089: "30-34", 93090: "35-39", 93091: "40-44", 93092: "45-49", 93093: "50-54",
    93094: "55-59", 93095: "60-64", 93096: "65-69", 93097: "70-74",
}


def fetch_populacao_idade(ano_censo: int = 2022) -> pd.DataFrame:
    """População por município × faixa quinquenal (5-74), Censo 2022. Em 2 lotes (<50k valores/call)."""
    cache = IBGE_DIR / f"populacao_idade_municipal_{ano_censo}.csv"
    if cache.exists():
        print(f"  [cache] {cache.name}")
        return pd.read_csv(cache, dtype={"CODMUN6": str})

    keys = list(GRUPOS_IDADE)
    frames = []
    for lote in (keys[:7], keys[7:]):
        codes = ",".join(map(str, lote))
        url = (f"https://apisidra.ibge.gov.br/values/t/9514/n6/all/v/93/p/{ano_censo}"
               f"/c2/6794/c287/{codes}")
        print(f"  → Sidra 9514: faixas {GRUPOS_IDADE[lote[0]]}…{GRUPOS_IDADE[lote[-1]]}")
        raw = _sidra_get(url)
        if not raw:
            raise RuntimeError(f"Sidra 9514 vazio (lote {lote})")
        frames.append(pd.DataFrame(raw[1:]))

    df = pd.concat(frames, ignore_index=True)
    df["POP"]     = pd.to_numeric(df["V"], errors="coerce")
    df["CODMUN6"] = df["D1C"].astype(str).str[:6]    # D1C=município, D5C=idade, D4C=sexo
    df["FAIXA"]   = df["D5C"].astype(int).map(GRUPOS_IDADE)
    pop = df.groupby(["CODMUN6", "FAIXA"], as_index=False)["POP"].sum()
    pop.to_csv(cache, index=False)
    print(f"     {pop['CODMUN6'].nunique():,} municípios | pop 5-74: {int(pop['POP'].sum()):,}")
    return pop


df_pop_idade = fetch_populacao_idade(2022)
print(f"\nPopulação por idade em: {IBGE_DIR}")

  [cache] populacao_idade_municipal_2022.csv

População por idade em: C:\Users\cerb3\OneDrive\Área de Trabalho\IESB\Tópicos BD\data\raw\ibge


## 4. PIB Municipal — IBGE Sidra Tabela 5938

**Tabela 5938** — Produto Interno Bruto dos Municípios.  
**Variável 37** — PIB total a preços correntes (R$ mil).

PIB per capita é calculado no notebook 2 dividindo pelo dado de população (Seção 3).

**Lag estrutural:** a contabilidade municipal só é fechada com 2 anos de defasagem. Em maio/2026 o último ano disponível é 2023. Para `ano=2024`, aplica-se fallback automático para o PIB do ano anterior disponível, marcado em `ANO_FONTE`.

In [7]:
def _fetch_pib_url(ano_fonte: int) -> list[dict]:
    url = f"https://apisidra.ibge.gov.br/values/t/5938/n6/all/v/37/p/{ano_fonte}"
    print(f"  → Sidra: {url}")
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    raw = r.json()
    return raw if len(raw) >= 2 else []


def fetch_pib_municipal(ano: int, max_lookback: int = 5) -> pd.DataFrame:
    """Busca PIB para `ano`. Se indisponível (lag IBGE), recua até `max_lookback` anos."""
    cache = IBGE_DIR / f"pib_municipal_{ano}.csv"
    if cache.exists():
        return pd.read_csv(cache, dtype={"CODMUN7": str, "CODMUN6": str})

    raw, ano_fonte = [], ano
    for tentativa in range(max_lookback + 1):
        ano_fonte = ano - tentativa
        raw = _fetch_pib_url(ano_fonte)
        if raw:
            if ano_fonte != ano:
                print(f"  ⚠️  PIB {ano} indisponível — usando {ano_fonte} como fallback")
            break
    if not raw:
        raise RuntimeError(f"PIB indisponível para {ano} mesmo após {max_lookback} anos de lookback")

    df = pd.DataFrame(raw[1:])
    df = df.rename(columns={"D1C": "CODMUN7", "D1N": "MUNICIPIO_NOME", "V": "PIB_RS_MIL"})
    df["PIB_RS_MIL"]  = pd.to_numeric(df["PIB_RS_MIL"], errors="coerce")
    df["ANO"]         = ano           # ano de referência
    df["ANO_FONTE"]   = ano_fonte     # origem real
    df["CODMUN7"]     = df["CODMUN7"].astype(str).str.strip()
    df["CODMUN6"]     = df["CODMUN7"].str[:6]
    df = df[["CODMUN7", "CODMUN6", "MUNICIPIO_NOME", "ANO", "ANO_FONTE", "PIB_RS_MIL"]]
    df.to_csv(cache, index=False)
    print(f"     {len(df):,} municípios | PIB total: R$ {df['PIB_RS_MIL'].sum()/1_000_000:,.1f} bi")
    return df


for ano in ANOS:
    fetch_pib_municipal(ano)

print(f"\nPIB em: {IBGE_DIR}")


PIB em: C:\Users\cerb3\OneDrive\Área de Trabalho\IESB\Tópicos BD\data\raw\ibge


## 5. Despesa em Saúde — SICONFI / Tesouro Nacional

**API DCA** (Demonstrações Contábeis Anuais), **Anexo I-E** (Despesas por Função).  
Função CID-10 das contas: **`10 - Saúde`** → linhas com despesas empenhadas, liquidadas, pagas e restos a pagar.

**Endpoint:** `https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca`

A API exige uma chamada por município/ano. Para acelerar, usamos `ThreadPoolExecutor` com 10 workers (~5 minutos para todo o Brasil em um ano).

In [8]:
SICONFI_URL = "https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca"


def listar_municipios_ibge() -> list[dict]:
    """Lista todos os municípios com seus códigos IBGE 7 dígitos via SICONFI."""
    cache = SICONFI_DIR / "municipios.json"
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))
    r = requests.get("https://apidatalake.tesouro.gov.br/ords/siconfi/tt/entes", timeout=60)
    r.raise_for_status()
    municipios = [e for e in r.json()["items"] if e.get("esfera") == "M"]
    cache.write_text(json.dumps(municipios, ensure_ascii=False), encoding="utf-8")
    return municipios


def fetch_despesa_saude_municipio(cod_ibge: int, ano: int) -> list[dict]:
    """Busca todas as linhas de Despesa por Função = '10 - Saúde' para um município/ano."""
    cache = SICONFI_DIR / f"_mun_{cod_ibge}_{ano}.json"
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))

    params = {"an_exercicio": ano, "no_anexo": "DCA-Anexo I-E", "id_ente": cod_ibge}
    try:
        r = requests.get(SICONFI_URL, params=params, timeout=60)
        r.raise_for_status()
        items = r.json().get("items", [])
    except Exception:
        items = []

    saude = [
        {
            "COD_IBGE":   x.get("cod_ibge"),
            "UF":         x.get("uf"),
            "ANO":        x.get("exercicio"),
            "COLUNA":     x.get("coluna"),
            "VALOR":      x.get("valor"),
            "POPULACAO":  x.get("populacao"),
        }
        for x in items if (x.get("conta") or "").strip() == "10 - Saúde"
    ]
    cache.write_text(json.dumps(saude, ensure_ascii=False), encoding="utf-8")
    return saude


def fetch_despesa_saude_ano(ano: int, max_workers: int = 10) -> pd.DataFrame:
    """Busca despesa em saúde de todos os municípios para um ano. Concorrente."""
    csv_path = SICONFI_DIR / f"despesa_saude_{ano}.csv"
    if csv_path.exists():
        print(f"  [cache] {csv_path.name}")
        return pd.read_csv(csv_path)

    municipios = listar_municipios_ibge()
    print(f"  Buscando despesa em saúde de {len(municipios)} municípios para {ano}...")
    t0 = time.time()
    todos = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch_despesa_saude_municipio, m["cod_ibge"], ano): m for m in municipios}
        feitos = 0
        for fut in as_completed(futures):
            todos.extend(fut.result())
            feitos += 1
            if feitos % 500 == 0:
                print(f"     {feitos}/{len(municipios)} ({(time.time()-t0):.0f}s)")

    df = pd.DataFrame(todos)
    df.to_csv(csv_path, index=False)
    # Limpa caches por município (já consolidados no CSV). missing_ok evita race condition
    # quando execução paralela ou re-run interrompido causa stale globs.
    for f in SICONFI_DIR.glob(f"_mun_*_{ano}.json"):
        f.unlink(missing_ok=True)
    print(f"  Concluído em {(time.time()-t0):.0f}s — {len(df):,} linhas → {csv_path.name}")
    return df


for ano in ANOS:
    fetch_despesa_saude_ano(ano)

print(f"\nSICONFI em: {SICONFI_DIR}")

  [cache] despesa_saude_2022.csv
  [cache] despesa_saude_2023.csv
  [cache] despesa_saude_2024.csv



SICONFI em: C:\Users\cerb3\OneDrive\Área de Trabalho\IESB\Tópicos BD\data\raw\siconfi


## 6. Atlas DH — IDH-M, Gini, Renda por quinto via API IPEA Data

**API:** `http://www.ipeadata.gov.br/api/odata4/`

Os dados do Atlas do Desenvolvimento Humano (PNUD/IPEA/FJP) — IDH-M, Gini, renda média por quinto, pobreza, analfabetismo, esperança de vida — estão **integralmente disponíveis na API do IPEA Data** sob códigos `ADH_*`. Usamos isso em vez do download manual em <atlasbrasil.org.br>.

**Ano de referência:** 2010 (último Censo com Atlas calculado; próxima rodada com Censo 2022 ainda não publicada).

**Tempo:** ~7 segundos (17 indicadores × 5.564 municípios em paralelo).

| Código IPEA | Coluna saída | Descrição |
|---|---|---|
| `ADH_IDHM` | `IDHM` | Índice de Desenvolvimento Humano Municipal |
| `ADH_IDHM_E` / `_L` / `_R` | `IDHM_E` / `IDHM_L` / `IDHM_R` | Sub-índices Educação / Longevidade / Renda |
| `ADH_GINI` | `GINI` | Coeficiente de Gini |
| `ADH_RDPC` | `RDPC` | Renda domiciliar per capita média |
| `ADH_RDPC1`–`RDPC4` | `RDPC1`–`RDPC4` | **Renda média do 1º ao 4º quinto mais pobre** |
| `ADH_RDPC5` | `RDPC5` | Renda média do quinto mais rico |
| `ADH_PMPOB` / `ADH_PIND` | `PMPOB` / `PIND` | % de pobres / extremamente pobres |
| `ADH_THEIL` | `THEIL` | Índice de Theil-L (desigualdade) |
| `ADH_ESPVIDA` | `ESPVIDA` | Esperança de vida ao nascer |
| `ADH_T_ANALF15M` | `T_ANALF15M` | Taxa de analfabetismo (15+ anos) |
| `ADH_T_AGUA` | `T_AGUA` | % com água encanada |

In [9]:
INDICADORES_IPEA = {
    "ADH_IDHM":       "IDHM",
    "ADH_IDHM_E":     "IDHM_E",
    "ADH_IDHM_L":     "IDHM_L",
    "ADH_IDHM_R":     "IDHM_R",
    "ADH_GINI":       "GINI",
    "ADH_RDPC":       "RDPC",
    "ADH_RDPC1":      "RDPC1",   # renda média do 1º quinto mais pobre
    "ADH_RDPC2":      "RDPC2",
    "ADH_RDPC3":      "RDPC3",
    "ADH_RDPC4":      "RDPC4",
    "ADH_RDPC5":      "RDPC5",   # renda média do quinto mais rico
    "ADH_PMPOB":      "PMPOB",
    "ADH_PIND":       "PIND",
    "ADH_THEIL":      "THEIL",
    "ADH_ESPVIDA":    "ESPVIDA",
    "ADH_T_ANALF15M": "T_ANALF15M",
    "ADH_T_AGUA":     "T_AGUA",
}
ANO_ATLAS = 2010


def fetch_ipea_municipios(serie_codigo: str, ano: int = 2010) -> list[dict]:
    """Busca uma série IPEA Data filtrada por município/ano."""
    url = f"http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='{serie_codigo}')"
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    items = r.json()["value"]
    return [d for d in items
            if "unic" in d.get("NIVNOME", "")
            and (d.get("VALDATA") or "")[:4] == str(ano)]


def fetch_atlas_dh_ipea(ano: int = 2010) -> pd.DataFrame:
    """Baixa todos os indicadores do Atlas via IPEA em paralelo. Cache local."""
    ATLAS_FILE = ATLAS_DIR / f"atlas_dh_ipea_{ano}.csv"
    if ATLAS_FILE.exists():
        return pd.read_csv(ATLAS_FILE, dtype={"CODMUN7": str, "CODMUN6": str})

    print(f"  Baixando {len(INDICADORES_IPEA)} indicadores via IPEA Data (paralelo)...")
    t0 = time.time()
    resultados = {}
    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = {ex.submit(fetch_ipea_municipios, cod, ano): cod
                   for cod in INDICADORES_IPEA}
        for fut in as_completed(futures):
            cod = futures[fut]
            resultados[cod] = fut.result()
            print(f"    {cod:<22} {INDICADORES_IPEA[cod]:<12} {len(resultados[cod])} municípios")

    # Pivota: 1 linha por município, 1 coluna por indicador
    registros: dict[str, dict] = {}
    for cod, items in resultados.items():
        nome = INDICADORES_IPEA[cod]
        for d in items:
            cod7 = d["TERCODIGO"]
            if cod7 not in registros:
                registros[cod7] = {"CODMUN7": cod7}
            registros[cod7][nome] = d["VALVALOR"]

    df = pd.DataFrame(list(registros.values()))
    df["CODMUN6"] = df["CODMUN7"].str[:6]
    cols = ["CODMUN7", "CODMUN6"] + list(INDICADORES_IPEA.values())
    df = df[[c for c in cols if c in df.columns]]
    df.to_csv(ATLAS_FILE, index=False)
    print(f"  Concluído em {time.time()-t0:.1f}s — {len(df):,} municípios → {ATLAS_FILE.name}")
    return df


df_atlas = fetch_atlas_dh_ipea(ANO_ATLAS)
print(f"Dimensões: {df_atlas.shape}")
df_atlas.head(3)

Dimensões: (5564, 19)


,CODMUN7,CODMUN6,IDHM,IDHM_E,IDHM_L,IDHM_R,GINI,RDPC,RDPC1,RDPC2,RDPC3,RDPC4,RDPC5,PMPOB,PIND,THEIL,ESPVIDA,T_ANALF15M,T_AGUA
0,1200203,120020,0.664,0.582,0.776,0.648,0.64,450.06,28.67,120.84,222.94,387.63,1493.27,34.51,18.08,0.74,71.58,18.52,79.14
1,1200336,120033,0.625,0.546,0.770,0.580,0.61,295.50,15.37,78.28,157.50,288.28,938.10,45.68,28.99,0.69,71.17,23.79,80.26
2,1200351,120035,0.501,0.361,0.726,0.479,0.59,157.27,3.33,42.64,97.49,168.20,474.49,63.82,39.31,0.51,68.55,34.38,50.76


## 7. Resumo dos arquivos extraídos

In [10]:
def listar(diretorio: Path, padrao: str = "*") -> list[tuple[str, str]]:
    arquivos = sorted(diretorio.glob(padrao))
    return [(f.name, f"{f.stat().st_size / 1024:.1f} KB") for f in arquivos if f.is_file()]


print(f"=== SIM ({SIM_DIR}) ===")
for nome, tam in listar(SIM_DIR, "*.parquet"):
    print(f"  {nome:<25} {tam}")

print(f"\n=== IBGE ({IBGE_DIR}) ===")
for nome, tam in listar(IBGE_DIR, "*.csv"):
    print(f"  {nome:<35} {tam}")

print(f"\n=== SICONFI ({SICONFI_DIR}) ===")
for nome, tam in listar(SICONFI_DIR, "*.csv"):
    print(f"  {nome:<35} {tam}")

print(f"\n=== Atlas ({ATLAS_DIR}) ===")
for nome, tam in listar(ATLAS_DIR, "*"):
    print(f"  {nome:<35} {tam}")

=== SIM (C:\Users\cerb3\OneDrive\Área de Trabalho\IESB\Tópicos BD\data\raw\sim) ===
  DOAC2022.parquet          296.3 KB
  DOAC2023.parquet          317.1 KB
  DOAC2024.parquet          331.6 KB
  DOAL2022.parquet          1290.0 KB
  DOAL2023.parquet          1282.9 KB
  DOAL2024.parquet          1337.0 KB
  DOAM2022.parquet          1177.5 KB
  DOAM2023.parquet          1223.0 KB
  DOAM2024.parquet          1236.6 KB
  DOAP2022.parquet          280.2 KB
  DOAP2023.parquet          293.0 KB
  DOAP2024.parquet          301.4 KB
  DOBA2022.parquet          5412.2 KB
  DOBA2023.parquet          5428.5 KB
  DOBA2024.parquet          5775.3 KB
  DOCE2022.parquet          3397.7 KB
  DOCE2023.parquet          3290.8 KB
  DOCE2024.parquet          3492.0 KB
  DODF2022.parquet          917.3 KB
  DODF2023.parquet          894.4 KB
  DODF2024.parquet          992.8 KB
  DOES2022.parquet          1533.6 KB
  DOES2023.parquet          1562.3 KB
  DOES2024.parquet          1642.7 KB
  DOGO2022.pa

---

**Próximo passo:** abra `02_transformacao.ipynb` para aplicar a Lista Brasileira de Causas Evitáveis (LBCE), agregar óbitos por município e construir a matriz de features para a regressão.